In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<b><font size="5" color="red" >ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연동
- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용하여 DB 액세스 가능한 디스크 기반 DB
- SQLite는 프로토타입 용도
- 프로젝트 단계 : 분석 - 설계 - 구현 - 테스트 - 고객에게 배포 - 유지보수
                SQLite  -  Oracle/MySql, MariaDB, mongoDB
- C 라이브러리
DB.BrowserforSQLite[https://sqlitebrowser.org/]
## 1.1 SQLite 패키지 load

In [ ]:
import sqlite3
sqlite3.sqlite_version

In [ ]:
import pandas as pd
pd.__version__

## 1.2 데이터 베이스 연결

In [ ]:
# DB 연결
conn = sqlite3.connect('data/ch15_example.db')
conn

In [ ]:
# 커서객체(SQL문 실행->결과 조회) 생성
cursor = conn.cursor()
cursor

In [ ]:
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

### insert, update, delete 전송

In [ ]:
cursor.execute('''
    INSERT INTO MEMBER VALUES ('홍길동', 20, 'h@h.com')
''')
print('수행 결과 행수 :', cursor.rowcount)

In [ ]:
sql = "INSERT INTO MEMBER VALUES ('무길동', 30, 'k@h.com')"
print(sql)
cursor.execute(sql)
print('수행 결과 행수 :', cursor.rowcount)

In [ ]:
conn.commit() # (반) conn.rollback() DML에서만 가능

select 전송(파라미터 X)

In [ ]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER
''')

In [ ]:
# SELECT 전송
#   select 문 실행결과를 받는 함수
#   cursor.fetchone() : 결과를 한행씩 받을 때 (튜플 홍길동, 21, h@h.com)
#   cursor.fetchall() : 결과를 모두 받을 때 (튜플 list)
#   cursor.fetchmany(n) : 결과를 n행 받을 때(튜플 list)
#   cursor.description : header 내용을 포함한 내용들 (list)

# INSERT, UPDATE, DELETE 전송 : cursor.rowcount
print(cursor.fetchall())

In [ ]:
print(cursor.fetchall())

In [ ]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER order by NEXTAGE
''')
members = cursor.fetchmany(3)
members

In [ ]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER order by NEXTAGE
''')
members = cursor.fetchone()
members

In [ ]:
cursor.fetchall()

In [ ]:
# 데이터 한줄씩 가져오기
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER order by NEXTAGE
''')
members = []
while True:
    member = cursor.fetchone()
    if member is None:
        break
    print(member)
    members.append({'name':member[0],'age':member[1],'email':member[2]})
members

In [ ]:
# 데이터 한줄씩 가져오기
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER order by NEXTAGE
''')
members = []
while True:
    member = cursor.fetchone()
    if member is None:
        break
    print(member)
    members.append({'name':member[0],'age':member[1],'email':member[2]})
import pandas as pd
pd.DataFrame(members)

In [ ]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER order by NEXTAGE
''')
members = cursor.fetchall()
df = pd.DataFrame(members,
                  columns= [description[0] for description in cursor.description])
df

In [ ]:
# select문을 수행한 필드 정보
cursor.description

In [ ]:
[description[0] for description in cursor.description]

## 1.3 SQL 구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우 있음)
- named(추천)

In [ ]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','김길동')")
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : qmark 방법
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
sql = "SELECT * FROM MEMBER WHERE NAME IN (?,?)"
name1 = input('검색할 이름은?')
name2 = input('검색할 다른 이름은?')
cursor.execute(sql, (name1, name2))
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : named 방법
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
sql = "SELECT * FROM MEMBER WHERE NAME IN (:name1,:name2)"
name1 = input('검색할 이름은?')
name2 = input('검색할 다른 이름은?')
cursor.execute(sql, {'name1': name1, 'name2': name2})
cursor.fetchall()

In [ ]:
# named 방법으로 데이터 입력하기
try:
    name = input('입력할 이름은?')
    age = int(input('입력할 나이는?'))
except ValueError:
    print('유효하지 않은 나이를 입력하시면 18세로 초기화')
    age = 18
finally:
    email = input('입력할 메일은?')
cursor.execute('INSERT INTO MEMBER VALUES (:name, :age, :email)',
               {'name': name, 'age': age, 'email':email})
conn.commit()
if cursor.rowcount ==1:
    print('입력성공')
else:
    print('입력실패')

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle

In [ ]:
import cx_Oracle
# conn 얻어오는 방법1
conn = cx_Oracle.connect('scott','tiger','localhost:1521/xe')
conn

In [ ]:
# conn 얻어오는 방법2
oracle_dsn = cx_Oracle.makedsn(host='localhost', port=1521, sid="xe")
conn = cx_Oracle.connect("scott","tiger",dsn=oracle_dsn)
conn

In [ ]:
cursor = conn.cursor()
cursor.execute('SELECT EMPNO, ENAME, JOB, MGR, SAL, COMM FROM EMP')
# cursor.execute('SELECT * FROM EMP')
emps = cursor.fetchall()
for emp in emps:
    print(emp)

In [ ]:
pd.DataFrame(emps,
             columns=[des[0] for des in cursor.description])